In [ ]:
# CELL 1 — DL-POC connection and run name
# Paste ONLY your existing private sf_options = {...} connection block above
# the check below. Obtain it from your working patient-split notebook.
# Credentials are deliberately absent from this file.
# Use the approved compute that successfully ran the patient-split notebook.

if "sf_options" not in globals():
    raise RuntimeError("Add your private sf_options connection setup at the top of this cell.")

sf_options_dl_poc = sf_options.copy()
sf_options_dl_poc.update({
    "sfDatabase": "DSVC_TAKEDA_TA_PRIVATE",
    "sfSchema": "DS_ML",
})
PREFIX = "TAK861_TX_READY_V63_DL_POC"
RUN_ID = "RUN_001"  # Use exactly the same value in training and evaluation.

import re
if not re.fullmatch(r"[A-Z][A-Z0-9_]{0,39}", RUN_ID):
    raise ValueError("RUN_ID must use uppercase letters, numbers and underscores.")
MODEL_TABLE = f"{PREFIX}_MODEL_{RUN_ID}"
EVALUATION_TABLE = f"{PREFIX}_EVAL_{RUN_ID}"
print("DL-POC run:", RUN_ID)


In [ ]:
# CELL 2 — Dependencies and self-contained Snowflake/input helpers
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
try:
    import torch
    import sklearn
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
except ModuleNotFoundError as error:
    raise RuntimeError(
        "This notebook needs PyTorch, scikit-learn, NumPy, pandas and matplotlib. "
        "Use an approved ML runtime or install the missing package on your approved compute."
    ) from error
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
"""Embedded by the delivery notebooks; no repository dependency at runtime."""
import base64
import hashlib
import io
import json
import math
import re
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd


def canonical_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False)


def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def read_sf(suffix):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", f"{PREFIX}_{suffix}").load())


def table_exists(table):
    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):
        raise ValueError("Use uppercase letters, numbers and underscores in table names.")
    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "
             f"WHERE TABLE_SCHEMA = 'DS_ML' AND TABLE_NAME = '{table}'")
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load().limit(1).count() > 0)


def checked_metadata(frame, include_split=False):
    columns = ["PATIENT_ID", "END_DT", "RESP"]
    if include_split:
        columns += ["SPLIT", "SPLIT_CONFIG"]
    out = frame.loc[:, columns].copy()
    if out.empty or out.isna().any().any():
        raise ValueError("Snapshot metadata must be nonempty and contain no nulls.")
    if not out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x)).all():
        raise ValueError("PATIENT_ID must retain its original nonempty string value.")
    dates = pd.to_datetime(out.END_DT, errors="raise")
    if dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():
        raise ValueError("END_DT must be a date without an intraday time/timezone.")
    out["END_DT"] = dates.dt.strftime("%Y-%m-%d")
    if not out.RESP.isin([0, 1]).all():
        raise ValueError("RESP must be exactly 0 or 1 before conversion.")
    out["RESP"] = out.RESP.astype("int64")
    if out.duplicated(["PATIENT_ID", "END_DT"]).any():
        raise ValueError("Duplicate patient/date keys in snapshot metadata.")
    if include_split:
        if set(out.SPLIT) != {"train", "validation", "test"}:
            raise ValueError("Expected the saved train, validation and test assignments.")
        if out.groupby("PATIENT_ID", observed=True).SPLIT.nunique().gt(1).any():
            raise ValueError("Patient overlap between splits.")
        if len(out.SPLIT_CONFIG.unique()) != 1:
            raise ValueError("The split contains inconsistent creation settings.")
        for _, part in out.groupby("SPLIT", observed=True):
            if set(part.RESP) != {0, 1}:
                raise ValueError("Each split must contain both response classes.")
    return out.sort_values(["PATIENT_ID", "END_DT"]).reset_index(drop=True)


def validate_metadata_pair(snapshot_frame, manifest_frame, features):
    source = checked_metadata(snapshot_frame)
    manifest = checked_metadata(manifest_frame, include_split=True)
    if not source.equals(manifest[["PATIENT_ID", "END_DT", "RESP"]]):
        raise ValueError("Frozen split and source snapshots differ in keys or labels.")
    creation = json.loads(manifest.SPLIT_CONFIG.iloc[0])
    # Match the feature-order hash created in the completed split notebook.
    feature_order_hash = hashlib.sha256(
        json.dumps(features, ensure_ascii=False).encode("utf-8")).hexdigest()
    if creation.get("feature_order_sha256") != feature_order_hash or creation.get("n_timesteps") != 12:
        raise ValueError("Feature order or timesteps differ from the frozen split.")
    return manifest, creation


def new_tensor_buffer(metadata, feature_count, seq_len=12):
    temporary = tempfile.TemporaryDirectory(prefix="dl_poc_")
    path = Path(temporary.name) / "counts.float32"
    X = np.memmap(path, dtype="<f4", mode="w+", shape=(len(metadata), seq_len, feature_count))
    return temporary, X


def fill_tensor(X, metadata, sequence_rows):
    """Place rows by canonical keys, independent of Spark partition order."""
    positions = {(r.PATIENT_ID, r.END_DT): i for i, r in enumerate(metadata.itertuples())}
    seen = np.zeros(len(metadata), dtype=bool)
    seq_len, feature_count = X.shape[1:]
    for row in sequence_rows:
        raw_date = row["END_DT"]
        if raw_date is None or row["PATIENT_ID"] is None:
            raise ValueError("Null monthly snapshot key.")
        date = pd.Timestamp(raw_date)
        if date.tzinfo is not None or date != date.normalize():
            raise ValueError("Monthly END_DT is not an exact date.")
        key = (row["PATIENT_ID"], date.strftime("%Y-%m-%d"))
        if key not in positions:
            raise ValueError("Unexpected monthly snapshot key.")
        i = positions[key]
        if seen[i] or row["RESP"] != metadata.RESP.iloc[i]:
            raise ValueError("Duplicate monthly snapshot or changed label.")
        sequence = row["SEQUENCE"]
        steps = [month["T"] for month in sequence]
        if len(sequence) != seq_len or any(t is None for t in steps):
            raise ValueError("Expected exactly 12 complete timesteps per snapshot.")
        # Check original values before any integer conversion.
        if sorted(steps) != list(range(seq_len)):
            raise ValueError("Timesteps must be unique integers 0 through 11.")
        sequence = sorted(sequence, key=lambda month: month["T"])
        values = np.asarray([month["V"] for month in sequence], dtype=np.float32)
        if values.shape != (seq_len, feature_count):
            raise ValueError("Monthly feature shape does not match the vocabulary.")
        if not np.isfinite(values).all() or (values < 0).any():
            raise ValueError("Counts must be finite, nonnegative float32 values with no nulls.")
        X[i] = values
        seen[i] = True
    if not seen.all():
        raise ValueError("Monthly data is missing original snapshots, including zero-activity sequences.")
    X.flush()


def input_fingerprints(X, metadata, features):
    tensor_hash = hashlib.sha256()
    tensor_hash.update(canonical_json(list(X.shape)).encode("utf-8"))
    for start in range(0, len(X), 128):
        tensor_hash.update(np.asarray(X[start:start + 128], dtype="<f4").tobytes(order="C"))
    records = [[str(r.PATIENT_ID), str(r.END_DT), int(r.RESP), str(r.SPLIT)]
               for r in metadata.itertuples()]
    return {"model_input_sha256": tensor_hash.hexdigest(),
            "snapshot_manifest_sha256": digest_json(records),
            "feature_names_sha256": digest_json(features),
            "split_config_sha256": digest_json(json.loads(metadata.SPLIT_CONFIG.iloc[0]))}


def load_inputs():
    from pyspark.sql import functions as F
    mapping = read_sf("FEATURE_MAP").orderBy("FEATURE_INDEX").collect()
    if not mapping or [r["FEATURE_INDEX"] for r in mapping] != list(range(len(mapping))):
        raise ValueError("Invalid feature indices.")
    features = [r["FEATURE_NAME"] for r in mapping]
    aliases = [r["FEATURE_COLUMN"] for r in mapping]
    if (aliases != [f"F{i:04d}" for i in range(len(features))]
            or not all(isinstance(f, str) and f for f in features)
            or len(set(features)) != len(features)):
        raise ValueError("Invalid feature names, aliases or order.")
    source = read_sf("SNAPSHOTS").select("PATIENT_ID", "END_DT", "RESP").toPandas()
    frozen = read_sf("PATIENT_SPLIT").select(
        "PATIENT_ID", "END_DT", "RESP", "SPLIT", "SPLIT_CONFIG").toPandas()
    metadata, creation = validate_metadata_pair(source, frozen, features)
    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()), len(features))
    if observed != (23151, 12447, 1345, 1028):
        raise ValueError(f"V63 population changed: snapshots/patients/positives/features = {observed}.")
    monthly = read_sf("TENSOR_MONTHLY")
    if set(monthly.columns) != set(["PATIENT_ID", "END_DT", "RESP", "TIME_STEP"] + aliases):
        raise ValueError("Monthly table columns differ from the frozen feature map.")
    print("Building the model input in a temporary driver file (about 1.06 GiB).", flush=True)
    grouped = (monthly.groupBy("PATIENT_ID", "END_DT", "RESP")
        .agg(F.collect_list(F.struct(
            F.col("TIME_STEP").alias("T"),
            F.array(*[F.when(F.col(name) >= 0, F.col(name).cast("float"))
                      .otherwise(F.lit(None).cast("float")) for name in aliases]).alias("V")
        )).alias("SEQUENCE"))
        .repartition(128))
    temporary, X = new_tensor_buffer(metadata, len(features))
    try:
        fill_tensor(X, metadata, grouped.toLocalIterator(prefetchPartitions=False))
        hashes = input_fingerprints(X, metadata, features)
        X.flags.writeable = False
    except BaseException:
        X._mmap.close()
        temporary.cleanup()
        raise
    y = metadata.RESP.to_numpy(dtype=np.float32)
    indices = {name: np.flatnonzero(metadata.SPLIT.to_numpy() == name)
               for name in ("train", "validation", "test")}
    print(f"Validated tensor shape {X.shape}; patient overlap = 0.", flush=True)
    return {"X": X, "y": y, "metadata": metadata, "features": features,
            "indices": indices, "hashes": hashes, "split_creation": creation,
            "temporary_directory": temporary}


def pack_artifacts(artifacts, chunk_size=50000):
    rows = []
    for name, blob in artifacts.items():
        encoded = base64.b64encode(blob).decode("ascii")
        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]
        digest = hashlib.sha256(blob).hexdigest()
        rows.extend((name, i, len(pieces), len(blob), digest, piece)
                    for i, piece in enumerate(pieces))
    return rows


def unpack_artifacts(rows, expected_names):
    groups = {}
    for row in rows:
        name, i, count, size, digest, payload = tuple(row)
        if any(value != int(value) for value in (i, count, size)):
            raise ValueError("Nonintegral artifact chunk metadata.")
        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))
    if set(groups) != set(expected_names):
        raise ValueError("Missing or unexpected saved artifacts.")
    result = {}
    for name, pieces in groups.items():
        pieces.sort(key=lambda p: p[0])
        count, size, digest = pieces[0][1:4]
        if (count < 1 or size < 0 or len(pieces) != count
                or [p[0] for p in pieces] != list(range(count))
                or any(p[1:4] != (count, size, digest) for p in pieces)):
            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")
        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)
        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:
            raise ValueError("Artifact length/hash mismatch.")
        result[name] = blob
    return result


ARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]


def read_artifacts(table, expected_names):
    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())
    return unpack_artifacts(rows, expected_names)


def save_artifacts(table, artifacts):
    # A matching existing result can be verified after an interrupted read-back.
    if table_exists(table):
        if read_artifacts(table, artifacts) != artifacts:
            raise FileExistsError("Destination contains different artifacts; choose a new RUN_ID.")
        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
        return
    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "
              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")
    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)
    (frame.write.format("snowflake").options(**sf_options_dl_poc)
     .option("dbtable", table).option("truncate_columns", "off")
     .mode("errorifexists").save())
    if read_artifacts(table, artifacts) != artifacts:
        raise ValueError("Saved artifact read-back differs from the completed run.")
    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")


In [ ]:
# CELL 3 — Read the frozen DL-POC inputs; validate identity and build raw-count tensor
# This does not calculate TEST predictions or TEST performance.
if "data" in globals():
    old_X = data.get("X")
    if old_X is not None and not old_X._mmap.closed:
        old_X._mmap.close()
        data["temporary_directory"].cleanup()
data = load_inputs()
display(data["metadata"].groupby("SPLIT", observed=True).agg(
    patients=("PATIENT_ID", "nunique"),
    snapshots=("RESP", "size"),
    positive_snapshots=("RESP", "sum"),
))


In [ ]:
# CELL 4 — Model, training functions and experiment settings
# Two encoder layers; mean pooling includes zero-activity months.
# Time order stays 0 (newest) through 11 (oldest).
# log1p is applied once per batch; no preprocessing statistics are fitted.
"""Small sequence classifier; zero-activity months remain real timesteps."""

from dataclasses import dataclass

import torch
from torch import nn


@dataclass(frozen=True)
class ModelConfig:
    input_dim: int
    seq_len: int = 12
    d_model: int = 128
    n_heads: int = 4
    encoder_layers: int = 2
    feedforward_dim: int = 256
    dropout: float = 0.2

    def __post_init__(self):
        sizes = (self.input_dim, self.seq_len, self.d_model, self.n_heads,
                 self.encoder_layers, self.feedforward_dim)
        if any(not isinstance(n, int) or isinstance(n, bool) or n <= 0 for n in sizes):
            raise ValueError("All model dimensions must be positive integers.")
        if self.d_model % self.n_heads:
            raise ValueError("d_model must be divisible by n_heads.")
        if not 0 <= self.dropout < 1:
            raise ValueError("dropout must be in [0, 1).")


class ClaimsTransformer(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.projection = nn.Linear(config.input_dim, config.d_model)
        self.position = nn.Parameter(torch.empty(1, config.seq_len, config.d_model))
        nn.init.normal_(self.position, std=0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=config.d_model, nhead=config.n_heads,
            dim_feedforward=config.feedforward_dim, dropout=config.dropout,
            activation="relu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=config.encoder_layers, enable_nested_tensor=False,
        )
        # TransformerEncoder clones the initial layer; initialize matrix weights
        # independently so the layers do not begin with identical weights.
        for encoder_layer in self.encoder.layers:
            for parameter in encoder_layer.parameters():
                if parameter.dim() > 1:
                    nn.init.xavier_uniform_(parameter)
        self.norm = nn.LayerNorm(config.d_model)
        self.head = nn.Sequential(
            nn.Linear(config.d_model, 64), nn.ReLU(),
            nn.Dropout(config.dropout), nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        expected = (self.config.seq_len, self.config.input_dim)
        if x.ndim != 3 or tuple(x.shape[1:]) != expected:
            raise ValueError(f"Expected [batch, {expected[0]}, {expected[1]}] input.")
        if not x.is_floating_point():
            raise TypeError("Transformer input must be floating point.")
        # A zero month is observed absence of activity, not padding. A causal
        # mask is unnecessary because every included month precedes the cutoff.
        hidden = self.projection(x) + self.position
        pooled = self.norm(self.encoder(hidden)).mean(dim=1)
        return self.head(pooled).squeeze(-1)

"""Classification metrics and a threshold selected only on VALIDATION."""

import numpy as np
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)


def _validate(y, probabilities):
    labels = np.asarray(y)
    raw_scores = np.asarray(probabilities)
    if np.iscomplexobj(raw_scores) or (
        raw_scores.dtype == object
        and any(isinstance(value, (complex, np.complexfloating)) for value in raw_scores.flat)
    ):
        raise ValueError("Probabilities must be real values, not complex numbers.")
    scores = np.asarray(raw_scores, dtype=float)
    if labels.ndim != 1 or scores.ndim != 1 or len(labels) != len(scores) or not len(labels):
        raise ValueError("Labels and probabilities must be aligned, nonempty 1-D arrays.")
    if not np.isin(labels, [0, 1]).all():
        raise ValueError("Labels must be binary 0/1.")
    if not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():
        raise ValueError("Probabilities must be finite and in [0, 1].")
    return labels.astype(np.int64), scores


def select_validation_threshold(y, probabilities) -> float:
    """Maximize VALIDATION F1; an exact tie uses the highest threshold.

    The caller must provide VALIDATION labels and scores, never TEST. Predictions
    are positive when score >= threshold. Equal scores are never split, and
    integer cross-products identify exact F1 ties without rounding ambiguity.
    """
    labels, scores = _validate(y, probabilities)
    if len(np.unique(labels)) != 2:
        raise ValueError("Threshold selection requires both VALIDATION classes.")
    order = np.argsort(scores, kind="stable")[::-1]
    ranked_scores = scores[order]
    true_positives = np.cumsum(labels[order], dtype=np.int64)
    group_ends = np.r_[np.flatnonzero(ranked_scores[:-1] != ranked_scores[1:]), len(labels) - 1]
    total_positives = int(labels.sum())
    best_numerator, best_denominator = 0, 1
    best_threshold = float(ranked_scores[0])
    for end in group_ends:
        # F1 = 2 TP / (number selected + total positives). Python integers
        # keep cross-products exact and avoid fixed-width integer overflow.
        numerator = 2 * int(true_positives[end])
        denominator = int(end) + 1 + total_positives
        if numerator * best_denominator > best_numerator * denominator:
            best_numerator, best_denominator = numerator, denominator
            best_threshold = float(ranked_scores[end])
        # Descending thresholds retain the highest cutoff on an exact tie.
    return best_threshold


def classification_metrics(y, probabilities, threshold: float) -> dict:
    labels, scores = _validate(y, probabilities)
    if not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("The fixed classification threshold must be in [0, 1].")
    predicted = (scores >= threshold).astype(np.int64)
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    return {
        "average_precision": float(average_precision_score(labels, scores)) if labels.sum() else None,
        "roc_auc": float(roc_auc_score(labels, scores)) if len(np.unique(labels)) == 2 else None,
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "recall": float(recall_score(labels, predicted, zero_division=0)),
        "f1": float(f1_score(labels, predicted, zero_division=0)),
        "threshold": float(threshold),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

"""Seed configuration and aggregate-only runtime provenance."""

import os
import platform
import random

import numpy as np
import sklearn
import torch


def seed_everything(seed: int) -> None:
    if not isinstance(seed, int) or isinstance(seed, bool) or not 0 <= seed < 2**32:
        raise ValueError("seed must be an integer in [0, 2**32).")
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def seed_worker(worker_id: int) -> None:
    """Use the DataLoader's seeded generator for each worker process."""
    del worker_id
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def resolve_device(device: str = "auto") -> torch.device:
    if device == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    resolved = torch.device(device)
    if resolved.type not in {"cpu", "cuda"}:
        raise ValueError("Supported devices are auto, cpu, or cuda[:index].")
    if resolved.type == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA requested but unavailable; set device='cpu'.")
    return resolved


def runtime_metadata() -> dict:
    return {
        "python_version": platform.python_version(),
        "numpy_version": str(np.__version__),
        "sklearn_version": str(sklearn.__version__),
        "torch_version": str(torch.__version__),
        "cuda_version": str(torch.version.cuda),
        "deterministic_algorithms_requested": torch.are_deterministic_algorithms_enabled(),
        "determinism_scope": "Best effort within a fixed device and software environment; unsupported operations warn.",
    }

"""Notebook-local training; consumes the verified memory-mapped tensor."""
from dataclasses import asdict
from datetime import datetime, timezone

from torch.utils.data import Dataset, DataLoader


class SequenceDataset(Dataset):
    def __init__(self, data, split):
        self.data = data
        self.indices = data["indices"][split]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        row = int(self.indices[index])
        x = torch.from_numpy(np.array(self.data["X"][row], dtype=np.float32, copy=True))
        return x, torch.tensor(float(self.data["y"][row]), dtype=torch.float32)


def make_loader(data, split, batch_size, seed, shuffle=False):
    return DataLoader(SequenceDataset(data, split), batch_size=batch_size,
                      shuffle=shuffle, num_workers=0, drop_last=False,
                      generator=torch.Generator().manual_seed(seed))


def checked_log1p(x):
    if not torch.isfinite(x).all() or (x < 0).any():
        raise ValueError("Expected finite nonnegative raw counts before log1p.")
    return torch.log1p(x)


def predict_loader(model, loader, device, criterion=None):
    model.eval()
    labels, scores, total_loss, count = [], [], 0.0, 0
    with torch.inference_mode():
        for x, y in loader:
            x = checked_log1p(x.to(device))
            y = y.to(device)
            logits = model(x)
            if not torch.isfinite(logits).all():
                raise ValueError("Nonfinite model predictions.")
            if criterion is not None:
                loss = criterion(logits, y)
                if not torch.isfinite(loss):
                    raise ValueError("Nonfinite validation loss.")
                total_loss += float(loss.item()) * len(y)
            count += len(y)
            labels.append(y.cpu().numpy())
            scores.append(torch.sigmoid(logits).cpu().numpy())
    return total_loss / count, np.concatenate(labels), np.concatenate(scores)


def train_run(data, model_config, settings, run_id):
    if tuple(data["X"].shape[1:]) != (model_config.seq_len, model_config.input_dim):
        raise ValueError("Architecture and tensor dimensions disagree.")
    for name in ("epochs", "patience", "batch_size"):
        if not isinstance(settings[name], int) or settings[name] <= 0:
            raise ValueError(f"{name} must be a positive integer.")
    for name in ("learning_rate", "grad_clip"):
        if not np.isfinite(settings[name]) or settings[name] <= 0:
            raise ValueError(f"{name} must be positive and finite.")
    for name in ("weight_decay", "min_delta"):
        if not np.isfinite(settings[name]) or settings[name] < 0:
            raise ValueError(f"{name} must be nonnegative and finite.")
    seed_everything(settings["seed"])
    device = resolve_device(settings["device"])
    class_counts = {}
    for name in ("train", "validation"):
        y = data["y"][data["indices"][name]]
        positives = int(y.sum())
        negatives = len(y) - positives
        if not positives or not negatives:
            raise ValueError(f"{name} requires both classes.")
        class_counts[name] = {"snapshots": len(y), "positives": positives, "negatives": negatives}
    pos_weight = class_counts["train"]["negatives"] / class_counts["train"]["positives"]
    model = ClaimsTransformer(model_config).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings["learning_rate"],
                                 weight_decay=settings["weight_decay"])
    train_loader = make_loader(data, "train", settings["batch_size"], settings["seed"], True)
    validation_loader = make_loader(data, "validation", settings["batch_size"], settings["seed"])
    best_ap, patience_reference = -np.inf, -np.inf
    without_progress, best_epoch, best_state = 0, None, None
    history = []
    print(f"Training on {device}; TRAIN positive weight = {pos_weight:.4f}.", flush=True)
    for epoch in range(1, settings["epochs"] + 1):
        model.train()
        loss_sum, count = 0.0, 0
        for x, y in train_loader:
            x, y = checked_log1p(x.to(device)), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite training loss.")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), settings["grad_clip"], error_if_nonfinite=True)
            optimizer.step()
            loss_sum += float(loss.item()) * len(y)
            count += len(y)
        val_loss, val_y, val_scores = predict_loader(model, validation_loader, device, criterion)
        val_ap = float(average_precision_score(val_y, val_scores))
        history.append({"epoch": epoch, "training_loss": loss_sum / count,
                        "validation_loss": val_loss, "validation_average_precision": val_ap,
                        "validation_roc_auc": float(roc_auc_score(val_y, val_scores))})
        if val_ap > best_ap:
            best_ap, best_epoch = val_ap, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if val_ap > patience_reference + settings["min_delta"]:
            patience_reference, without_progress = val_ap, 0
        else:
            without_progress += 1
        print(f"Epoch {epoch:02d}: train loss={loss_sum/count:.5f}; "
              f"validation loss={val_loss:.5f}; validation AP={val_ap:.5f}", flush=True)
        if without_progress >= settings["patience"]:
            print(f"Early stopping; restoring epoch {best_epoch}.", flush=True)
            break
    model.load_state_dict(best_state, strict=True)
    _, val_y, val_scores = predict_loader(model, validation_loader, device, criterion)
    threshold = select_validation_threshold(val_y, val_scores)
    summary = {"run_id": run_id, "training_complete": True,
               "completed_at_utc": datetime.now(timezone.utc).isoformat(),
               "best_epoch": int(best_epoch), "epochs_completed": len(history),
               "best_validation_average_precision": best_ap,
               "validation_threshold": threshold,
               "validation_metrics": classification_metrics(val_y, val_scores, threshold),
               "train_pos_weight": float(pos_weight), "class_counts": class_counts,
               "model_parameter_count": sum(p.numel() for p in model.parameters()),
               "model_config": asdict(model_config), "training_settings": dict(settings),
               "input_hashes": data["hashes"], "resolved_device": str(device),
               "preprocessing": "per-batch log1p of raw counts; no fitted preprocessing",
               "test_inference_performed": False,
               "source_review": "Structural checks only; clinical cutoff, outcome-event exclusion and historical claims availability are not certified by these notebooks.",
               "runtime": runtime_metadata()}
    payload = {"format_version": 1, "training_complete": True, "run_id": run_id,
               "model_state_dict": best_state, "model_config": asdict(model_config),
               "training_settings": dict(settings), "input_hashes": data["hashes"],
               "feature_names": list(data["features"]), "time_steps": list(range(model_config.seq_len)),
               "tensor_shape": [int(v) for v in data["X"].shape],
               "transform": "log1p", "validation_threshold": float(threshold),
               "selection_metric": "validation_average_precision", "best_epoch": int(best_epoch),
               "best_validation_average_precision": best_ap}
    buffer = io.BytesIO()
    torch.save(payload, buffer)
    return buffer.getvalue(), summary, pd.DataFrame(history)


def load_verified_model(blob, data, run_id, device="auto"):
    payload = torch.load(io.BytesIO(blob), map_location="cpu", weights_only=True)
    if (not isinstance(payload, dict) or payload.get("format_version") != 1
            or payload.get("training_complete") is not True or payload.get("run_id") != run_id):
        raise ValueError("Checkpoint is incomplete, unsupported, or belongs to another run.")
    if payload.get("input_hashes") != data["hashes"]:
        raise ValueError("Tensor content, keys, labels, feature order or frozen split changed after training.")
    if (payload.get("feature_names") != data["features"]
            or payload.get("time_steps") != list(range(data["X"].shape[1]))
            or payload.get("tensor_shape") != list(data["X"].shape)):
        raise ValueError("Checkpoint tensor layout differs from current inputs.")
    threshold = payload.get("validation_threshold")
    if threshold is None or not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("Missing valid frozen validation threshold.")
    if payload.get("transform") != "log1p":
        raise ValueError("Unknown preprocessing; evaluation stopped.")
    model = ClaimsTransformer(ModelConfig(**payload["model_config"]))
    model.load_state_dict(payload["model_state_dict"], strict=True)
    resolved = resolve_device(device)
    model.to(resolved).eval()
    return model, payload, resolved


model_config = ModelConfig(input_dim=len(data["features"]), seq_len=12,
    d_model=128, n_heads=4, encoder_layers=2, feedforward_dim=256, dropout=0.2)
training_settings = {
    "seed": 42, "epochs": 20, "patience": 5, "min_delta": 1e-4,
    "batch_size": 64, "learning_rate": 1e-3, "weight_decay": 1e-4,
    "grad_clip": 1.0, "device": "auto",
}
display(pd.DataFrame([asdict(model_config)]))
display(pd.DataFrame([training_settings]))


In [ ]:
# CELL 5 — Train on TRAIN, select the checkpoint and threshold using VALIDATION
# Highest validation average precision selects the checkpoint (earliest exact tie).
# Maximum validation F1 selects the threshold (highest cutoff on an exact tie).
# TEST is never scored in this cell.
if data["X"]._mmap.closed:
    raise RuntimeError("Temporary input was released. Rerun cell 3 before training.")
if table_exists(MODEL_TABLE):
    raise FileExistsError("This RUN_ID already has a saved model. Choose a NEW RUN_ID for a new experiment.")
checkpoint_bytes, training_summary, history = train_run(
    data, model_config, training_settings, RUN_ID)
assert training_summary["test_inference_performed"] is False
print("Training complete. Run cells 6 and 7 to inspect and save this completed model.")


In [ ]:
# CELL 6 — Inspect TRAIN/VALIDATION results only
fields = ["run_id", "model_parameter_count", "epochs_completed", "best_epoch",
          "best_validation_average_precision", "train_pos_weight",
          "validation_threshold", "resolved_device", "test_inference_performed"]
display(pd.DataFrame([{key: training_summary[key] for key in fields}]))
display(pd.DataFrame([training_summary["validation_metrics"]]))
display(history)
history_figure, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].plot(history.epoch, history.training_loss, label="TRAIN")
axes[0].plot(history.epoch, history.validation_loss, label="VALIDATION")
axes[0].set(xlabel="Epoch", ylabel="Weighted binary cross-entropy", title="Loss")
axes[0].legend()
axes[1].plot(history.epoch, history.validation_average_precision, marker="o")
axes[1].axvline(training_summary["best_epoch"], color="gray", linestyle="--")
axes[1].set(xlabel="Epoch", ylabel="Average precision", title="VALIDATION checkpoint selection")
plt.show()


In [ ]:
# CELL 7 — Save the completed model and aggregate training outputs in Snowflake
# Retrying THIS cell after a connection failure does not retrain the model.
TRAINING_ARTIFACT_NAMES = {"checkpoint.pt", "training_summary.json", "training_history.csv", "training_history.png"}
if training_summary["run_id"] != RUN_ID:
    raise ValueError("RUN_ID changed after training; restore the completed run's original RUN_ID.")
image_buffer = io.BytesIO()
history_figure.savefig(image_buffer, format="png", dpi=150)
training_artifacts = {
    "checkpoint.pt": checkpoint_bytes,
    "training_summary.json": canonical_json(training_summary).encode("utf-8"),
    "training_history.csv": history.to_csv(index=False).encode("utf-8"),
    "training_history.png": image_buffer.getvalue(),
}
save_artifacts(MODEL_TABLE, training_artifacts)
print("Use RUN_ID =", repr(RUN_ID), "in the evaluation notebook after selecting this run on validation.")


In [ ]:
# CELL 8 — Release temporary tensor storage after the verified save
if read_artifacts(MODEL_TABLE, TRAINING_ARTIFACT_NAMES) != training_artifacts:
    raise ValueError("Saved artifacts are not verified; temporary inputs have been retained.")
if not data["X"]._mmap.closed:
    data["X"]._mmap.close()
    data["temporary_directory"].cleanup()
print("Training stage complete. The completed model is durable in Snowflake.")
print("Next: 04_transformer_evaluation_DL_POC, RUN_ID =", repr(RUN_ID))
